In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Separate features (X) and target (y)
# Drop 'id' as it has no predictive power
X = train_df.drop(columns=['id', 'Irrigation_Need'])  # Drop target and id
y = train_df['Irrigation_Need']  # Target variable

# Preprocessing
# Encode the target variable (Low, Medium, High -> 0, 1, 2)
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

# Identify categorical columns and encode them for the models
categorical_cols = X.select_dtypes(include=['object']).columns

# Simple Ordinal Encoding for starter models
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training set: {X_train.shape}, Validation set: {X_val.shape}")


Training set: (504000, 19), Validation set: (126000, 19)


In [3]:
# bagging model (Random Forest)
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

print("--- Starting Grid Search for Random Forest ---")

# Define a small grid of parameters to test
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [20, 40, None],
    'min_samples_split': [5, 10]
}

# Set up the Grid Search
# scoring='balanced_accuracy' is crucial for your Kaggle metric
rf_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_grid=rf_param_grid,
    scoring='balanced_accuracy',
    cv=5,
    verbose=1
)

# fit the grid search
rf_grid_search.fit(X_train, y_train)

print(f"\nBest Random Forest Parameters: {rf_grid_search.best_params_}")
print(f"Best CV Balanced Accuracy: {rf_grid_search.best_score_:.4f}")

# You can now use the best model directly to predict
best_rf_model = rf_grid_search.best_estimator_
rf_preds = best_rf_model.predict(X_val)

--- Starting Grid Search for Random Forest ---
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Best Random Forest Parameters: {'max_depth': 20, 'min_samples_split': 10, 'n_estimators': 100}
Best CV Balanced Accuracy: 0.9616


In [7]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

print("--- Preparing Test Data & Generating Submission ---")

# Load the Kaggle test data
test_df = pd.read_csv('test.csv')

# Define X_test by dropping the ID column
X_test = test_df.drop(columns=['id'])

# Preprocess X_test (Encode the categorical text columns into numbers)
categorical_cols = X_test.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    X_test[col] = le.fit_transform(X_test[col].astype(str))

# Grab the tuned model from your Grid Search
best_rf_model = rf_grid_search.best_estimator_

# Fit on the FULL training dataset
best_rf_model.fit(X, y_encoded)

# Make predictions on our newly defined X_test
rf_preds_encoded = best_rf_model.predict(X_test) 

# Reverse the Label Encoding to get "Low", "Medium", "High"
rf_preds_text = target_encoder.inverse_transform(rf_preds_encoded)

# Build the Kaggle-formatted DataFrame
submission_rf = pd.DataFrame({
    "id": test_df["id"],
    "Irrigation": rf_preds_text
})

# Save to CSV
submission_rf.to_csv("S6E4_rf_submission.csv", index=False)
print("Success! Random Forest submission saved!")

--- Preparing Test Data & Generating Submission ---
Success! Random Forest submission saved!
